# 순서 섞기 실험 (논문 Table 5): Transformer 쪽

**목적.** 논문은 환율 데이터에서 입력 순서를 섞어도 FEDformer, Autoformer, Informer의 오차가 거의 변하지 않는다고 보고한다(평균 −0.12% ~ +1.12%). Linear 쪽은 이미 재현했다(입력 96에서 +28%, +45%). Transformer 쪽도 같은 결과가 나오는지 직접 확인한다.

| 모델 | 논문: 전체 섞기 | 논문: 앞뒤 절반 바꾸기 |
|---|---|---|
| FEDformer | −0.09% | +0.20% |
| Autoformer | +0.09% | +1.12% |
| Informer | −0.12% | −0.18% |

**설정.** 공식 저장소의 환율 Transformer 설정(입력 96, label_len 48, 1 epoch)으로 원래 순서에서 한 번 학습한 뒤, 테스트 입력만 다섯 가지로 넣어 채점한다. 예측 길이 96·192·336·720, 시드 2021·2022·2023.

- `none` 그대로 · `shuf` 인코더 입력만 섞음 · `half` 인코더 입력의 앞뒤 절반을 바꿈
- `shuf_all`, `half_all` 디코더에 들어가는 시작 구간(입력의 마지막 48개)도 섞인 입력에서 가져옴

Transformer 계열은 인코더 입력 말고도 디코더에 입력의 마지막 48개를 원래 순서로 한 번 더 넣는다. 그래서 인코더만 섞으면 순서 정보가 디코더로 새어 들어갈 수 있다. 논문이 어느 쪽으로 했는지는 밝혀져 있지 않아서 둘 다 잰다.

**실행.** 런타임 **GPU → T4** → **모두 실행**. 예상 30분~1시간. 끝나면 맨 아래 셀의 출력을 채팅에 붙여 주세요. 중간에 끊겨도 다시 실행하면 끝난 칸은 건너뛴다(같은 런타임일 때).

In [ ]:
# 0. GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1. 코드와 데이터 준비
import os, subprocess
os.chdir('/content')
subprocess.run('pip -q install einops gdown', shell=True)  # FEDformer needs einops
if not os.path.exists('LTSF-Linear'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/cure-lab/LTSF-Linear.git'], check=True)
os.makedirs('/content/data', exist_ok=True)
if not os.path.exists('/content/data/exchange_rate.csv'):
    subprocess.run('gdown -q --folder '
                   'https://drive.google.com/drive/folders/1ZOYpTUa82_jCcxIdTmyr0LXQfvaM9vIy -O /content/raw_data && '
                   'find /content/raw_data -name "exchange_rate.csv" -exec cp -n {} /content/data/ \;', shell=True, check=True)
!git -C /content/LTSF-Linear log --oneline -1
!wc -l /content/data/exchange_rate.csv

In [ ]:
%%writefile /content/shuffle_formers.py
# 2. 채점 스크립트 (학습은 저장소 코드 그대로, 테스트만 다섯 가지로)
"""Table 5 (order shuffling) for the Transformer models, test-time perturbation only.

Usage (from anywhere):
  python shuffle_formers.py <repo_dir> <model> <pred_len> <seed> <dataset_dir>
  model: Autoformer | Informer | Transformer | FEDformer

The model is trained once on the normal order with the repository's exchange-rate
settings (seq_len 96, label_len 48, 1 epoch for the Formers), then the test set is scored
five ways:
  none      : as is
  shuf      : encoder input shuffled in time, decoder start tokens left as they are
  half      : encoder input halves swapped, decoder start tokens left as they are
  shuf_all  : as shuf, and the decoder start tokens are taken from the shuffled input
  half_all  : as half, and the decoder start tokens are taken from the swapped input
Prints lines 'RESULT,<model>,<pred_len>,<seed>,<mode>,<mse>,<mae>'.
"""
import os
import sys

repo, model, P, seed, data_dir = sys.argv[1], sys.argv[2], int(sys.argv[3]), int(sys.argv[4]), sys.argv[5]
fed = model == 'FEDformer'
root = os.path.join(repo, 'FEDformer') if fed else repo
os.chdir(root)
sys.path.insert(0, root)

import numpy as np
import torch
import exp.exp_main as em

if not hasattr(np, 'Inf'):  # NumPy 2 removed np.Inf, which the repository still uses
    np.Inf = np.inf

MODES = ['none', 'shuf', 'half', 'shuf_all', 'half_all']


def perturb(x, mode, g):
    if mode.startswith('shuf'):
        idx = torch.randperm(x.shape[1], generator=g).to(x.device)
        return x[:, idx, :]
    if mode.startswith('half'):
        h = x.shape[1] // 2
        return torch.cat([x[:, h:, :], x[:, :h, :]], dim=1)
    return x


def test(self, setting, test=0):
    _, loader = self._get_data(flag='test')
    if test:
        self.model.load_state_dict(torch.load(os.path.join('./checkpoints/' + setting, 'checkpoint.pth')))
    self.model.eval()
    a = self.args
    for mode in MODES:
        g = torch.Generator().manual_seed(seed)
        se = ae = 0.0
        cnt = 0
        with torch.no_grad():
            for bx, by, bxm, bym in loader:
                bx = bx.float().to(self.device); by = by.float().to(self.device)
                bxm = bxm.float().to(self.device); bym = bym.float().to(self.device)
                px = perturb(bx, mode, g)
                if 'Linear' in a.model:
                    out = self.model(px)
                else:
                    start = px[:, -a.label_len:, :] if mode.endswith('_all') else by[:, :a.label_len, :]
                    dec = torch.cat([start, torch.zeros_like(by[:, -a.pred_len:, :])], dim=1).float()
                    out = self.model(px, bxm, dec, bym)
                    if isinstance(out, (tuple, list)):
                        out = out[0]
                f = -1 if a.features == 'MS' else 0
                d = (out[:, -a.pred_len:, f:] - by[:, -a.pred_len:, f:]).double()
                se += float((d ** 2).sum()); ae += float(d.abs().sum()); cnt += d.numel()
        print(f'RESULT,{model},{P},{seed},{mode},{se / cnt:.6f},{ae / cnt:.6f}', flush=True)


em.Exp_Main.test = test

argv = ['--is_training', '1', '--root_path', data_dir, '--data_path', 'exchange_rate.csv',
        '--data', 'custom', '--features', 'M', '--seq_len', '96', '--pred_len', str(P),
        '--enc_in', '8', '--des', 'Exp', '--itr', '1', '--num_workers', '0']
argv += ['--label_len', '48', '--e_layers', '2', '--d_layers', '1', '--factor', '3',
         '--dec_in', '8', '--c_out', '8', '--train_epochs', '1', '--model', model]
argv += ['--task_id', f'sh_{P}_{seed}'] if fed else ['--model_id', f'sh_{P}_{seed}']
sys.argv = ['run.py' if fed else 'run_longExp.py'] + argv
src = open(sys.argv[0]).read()
src = src.replace('fix_seed = 2021', f'fix_seed = {seed}')
exec(compile(src, sys.argv[0], 'exec'), {'__name__': '__main__'})

## 3. 실행
모델 3개 × 예측 길이 4개 × 시드 3개 = 36번 학습. 결과는 `/content/shuffle_formers.csv`에 한 줄씩 쌓인다.

In [ ]:
import re, time, pandas as pd
OUT = '/content/shuffle_formers.csv'
done = set()
if os.path.exists(OUT):
    d = pd.read_csv(OUT); done = set(zip(d.model, d.pred_len, d.seed))
else:
    open(OUT, 'w').write('model,pred_len,seed,mode,mse,mae\n')
os.makedirs('/content/logs', exist_ok=True)
for model in ['FEDformer', 'Autoformer', 'Informer']:
    for P in [96, 192, 336, 720]:
        for seed in [2021, 2022, 2023]:
            if (model, P, seed) in done:
                continue
            t = time.time(); log = f'/content/logs/{model}_{P}_{seed}.log'
            r = subprocess.run(['python', '-u', '/content/shuffle_formers.py', '/content/LTSF-Linear', model, str(P), str(seed), '/content/data/'],
                               capture_output=True, text=True)
            open(log, 'w').write(r.stdout + r.stderr)
            rows = [l[7:] for l in r.stdout.splitlines() if l.startswith('RESULT,')]
            if len(rows) != 5:
                print(f'{model} {P} {seed}: 실패, 로그 끝부분:'); print((r.stdout + r.stderr)[-1500:]); continue
            open(OUT, 'a').write('\n'.join(rows) + '\n')
            print(f'{model} T={P} seed={seed} 완료 ({time.time() - t:.0f}초)')
            subprocess.run('rm -rf /content/LTSF-Linear/checkpoints /content/LTSF-Linear/FEDformer/checkpoints', shell=True)

## 4. 결과를 채팅에 붙여 주세요
아래 셀 출력을 그대로 복사해 주시면 됩니다.

In [ ]:
d = pd.read_csv('/content/shuffle_formers.csv')
base = d[d['mode'] == 'none'].set_index(['model', 'pred_len', 'seed']).mse
d['change_%'] = [100 * (r.mse / base[(r.model, r.pred_len, r.seed)] - 1) for r in d.itertuples()]
print(d.pivot_table(index='model', columns='mode', values='change_%', aggfunc='mean').round(2).to_string())
print()
print(open('/content/shuffle_formers.csv').read())